# Follow an MLP update before training
# MLP 학습 한 번을 따라가기

Extend the reactor to consecutive first-order reactions A → B → C. Predict all three outlet concentrations from T, tau, and feed. Understand how the network generates a prediction, how an error becomes a gradient, and how scaling affects a full training run.

반응을 A → B → C의 1차 연속반응으로 늘립니다. T·tau·feed에서 세 출구 농도를 예측합니다. 신경망이 예측을 만드는 과정, 오차가 gradient로 전달되는 과정, scaling이 전체 학습에 미치는 영향을 확인합니다.

## Setup / 실행 준비

Extract the complete lab ZIP and run this notebook in that folder, with the scripts and supplied models beside it. Install requirements.txt into this notebook’s Python environment. The cells save new results under guided-results/.

실습 ZIP 전체를 풀고 스크립트·제공 모델과 같은 폴더에서 notebook을 실행합니다. Notebook의 Python 환경에 requirements.txt를 설치합니다. 새 결과는 guided-results/에 저장됩니다.

```text
python -m pip install -r requirements.txt
```

## 1. Build three reference labels / 1. 기준 농도 세 개 만들기

At 330 K, k1=0.2/min and k2=0.1/min. Solve A first, then B using the A production term, then C using B. This triangular dependency explains the order of the three assignment statements.

330 K에서 k1=0.2/min, k2=0.1/min입니다. A를 먼저 구하고, 그 A로 B 생성량을 구한 뒤 B로 C를 구합니다. 이 의존 관계 때문에 대입문도 A, B, C 순서입니다.

In [1]:
from pathlib import Path
import numpy as np
import torch
import week02_lab as lab
xphysical = np.array([[330., 5., 1.2]])
ca = 1.2/(1+.2*5)
cb = .2*5*ca/(1+.1*5)
cc = .1*5*cb
print("CA, CB, CC:", ca, cb, cc)
np.testing.assert_allclose(lab.steady_state(xphysical), [[ca, cb, cc]])
np.testing.assert_allclose([ca, cb, cc], [.6, .4, .2])

CA, CB, CC: 0.6 0.39999999999999997 0.19999999999999998


The concentrations sum to 1.2 mol/L. The later output layer predicts all three together, so this equality is not automatically enforced by an ordinary MLP.

세 농도의 합은 1.2 mol/L입니다. 이후 출력층은 세 농도를 함께 예측하므로 일반 MLP가 이 등식을 자동으로 만족하지는 않습니다.

## 2. Inspect a tiny forward calculation / 2. 작은 신경망의 forward 확인하기

Use a two-neuron hidden layer to make each operation visible. The example input and target are dimensionless. Multiply the input by W1, apply ReLU, and combine hidden outputs with W2. ReLU sets negative preactivations to zero.

계산이 보이도록 은닉 뉴런 두 개를 사용합니다. 이 예제의 입력과 target은 무차원입니다. W1을 곱하고 ReLU를 적용한 뒤 W2로 결합합니다. ReLU는 음수인 preactivation을 0으로 바꿉니다.

In [2]:
x = torch.tensor([1., 0., -1.], dtype=torch.float64)
target = torch.tensor([0., 1., 0.], dtype=torch.float64)
w1 = torch.tensor([[1.,0.,0.],[0.,0.,1.]], dtype=torch.float64, requires_grad=True)
b1 = torch.zeros(2, dtype=torch.float64, requires_grad=True)
w2 = torch.tensor([[1.,0.],[0.,1.],[1.,1.]], dtype=torch.float64, requires_grad=True)
b2 = torch.zeros(3, dtype=torch.float64, requires_grad=True)
a = w1@x+b1
h = torch.relu(a)
prediction = w2@h+b2
error = prediction-target
loss = .5*(error**2).sum()
print("a, h, prediction, error, loss:", a.tolist(), h.tolist(), prediction.tolist(), error.tolist(), loss.item())
assert loss.item() == 1.5

a, h, prediction, error, loss: [1.0, -1.0] [1.0, 0.0] [1.0, 0.0, 1.0] [1.0, -1.0, 1.0] 1.5


a=[1,−1] becomes h=[1,0]. The prediction [1,0,1] minus target [0,1,0] gives [1,−1,1]. Half of 1+1+1 is loss 1.5. The second hidden unit contributes nothing on this input.

a=[1,−1]은 h=[1,0]이 됩니다. 예측 [1,0,1]에서 target [0,1,0]을 빼면 [1,−1,1]입니다. 1+1+1의 절반이 loss 1.5입니다. 이 입력에서는 두 번째 은닉 뉴런이 출력에 기여하지 않습니다.

## 3. Trace the gradient and parameter update / 3. Gradient와 파라미터 갱신 추적하기

For half squared error, the output derivative is the residual. Multiplying by W2 transpose gives the hidden derivative; the ReLU gate then removes the second component. The first hidden derivative is 2, so its weight gradient is 2 times x. Move opposite to the gradient with learning rate 0.1.

제곱 오차의 절반을 쓰면 출력 미분은 오차 벡터입니다. W2의 전치를 곱해 은닉층으로 전달한 뒤 ReLU의 비활성 성분을 0으로 만듭니다. 첫 은닉 미분이 2이므로 첫 가중치 행의 gradient는 2x입니다. 학습률 0.1로 gradient의 반대 방향으로 이동합니다.

In [3]:
loss.backward()
print("dW1:", w1.grad.tolist(), "db1:", b1.grad.tolist())
print("dW2:", w2.grad.tolist(), "db2:", b2.grad.tolist())
np.testing.assert_allclose(w1.grad, [[2,0,-2],[0,0,0]])
with torch.no_grad():
    for parameter in (w1,b1,w2,b2):
        parameter -= .1*parameter.grad
    after = w2@torch.relu(w1@x+b1)+b2
    after_loss = .5*((after-target)**2).sum()
print("updated prediction and loss:", after.tolist(), after_loss.item())
check = lab.backprop_example()
np.testing.assert_allclose(after.tolist(), check["prediction_after"])
assert after_loss.item() < loss.item()

dW1: [[2.0, 0.0, -2.0], [0.0, 0.0, -0.0]] db1: [2.0, 0.0]
dW2: [[1.0, 0.0], [-1.0, -0.0], [1.0, 0.0]] db2: [1.0, -1.0, 1.0]
updated prediction and loss: [0.2600000000000001, 0.14, 0.2600000000000001] 0.43740000000000007


The updated prediction is [0.26,0.14,0.26] and loss is 0.4374. This one-sample decrease checks the update direction; it does not establish test performance. In repeated training, gradients must be cleared before the next backward call, as fit_mlp does with optimizer.zero_grad().

갱신 후 예측은 [0.26,0.14,0.26], loss는 0.4374입니다. 한 표본의 loss 감소로 갱신 방향을 확인할 수 있지만 test 성능은 따로 평가합니다. 반복 학습에서는 fit_mlp의 optimizer.zero_grad()처럼 다음 backward 전에 gradient를 지웁니다.

## 4. Scale, train, and read the physical errors / 4. Scaling·학습·물리 단위 오차 읽기

The actual network is 3→32→32→3: (3×32+32)+(32×32+32)+(32×3+3)=1283 parameters. Inputs use declared bounds; output means and standard deviations come only from training rows. Validation chooses the saved epoch; test rows assess that fixed model.

실제 신경망은 3→32→32→3으로 파라미터가 (3×32+32)+(32×32+32)+(32×3+3)=1283개입니다. 입력은 선언한 영역으로, 출력은 train 평균·표준편차로 scaling합니다. Validation으로 저장할 epoch를 고른 뒤 고정 모델을 test로 평가합니다.

In [4]:
splits = lab.generate_steady_splits(seed=42)
tx, ty = splits["train"]
print("input scaled first row:", (tx[0]-(lab.LOWER+lab.UPPER)/2)/((lab.UPPER-lab.LOWER)/2))
print("output mean, scale:", ty.mean(0), ty.std(0))
assert sum(p.numel() for p in lab.make_model(3).parameters()) == 1283
report = lab.run_steady(Path("guided-results"), seed=42, epochs=1400)
model, scaling = lab.load_model(Path("guided-results"), "steady-model")
vx, truth = splits["test"]
pred = lab.predict(model, scaling, vx)
print("first three reference / prediction rows:", np.column_stack([truth[:3], pred[:3]]))
manual_rmse = np.sqrt(np.mean((pred-truth)**2, axis=0))
print("physical RMSE:", manual_rmse)
np.testing.assert_allclose(manual_rmse, report["scaled_output_model"]["rmse_mol_L"])

input scaled first row: [ 0.5479121  -0.12224312  0.71719584]
output mean, scale: [0.59700654 0.31580225 0.24053176] [0.28303718 0.11000633 0.21743125]
epoch 400: train=0.000486822, validation=0.000719781
epoch 800: train=0.000209511, validation=0.000461487
epoch 1200: train=0.000215109, validation=0.00040381
epoch 400: train=2.87667e-05, validation=3.90648e-05
epoch 800: train=1.09174e-05, validation=2.15381e-05
epoch 1200: train=8.64387e-06, validation=1.91342e-05
{
  "scaled_output_model": {
    "rmse_mol_L": [
      0.005275124114926506,
      0.0029403857147795205,
      0.004423193366032517
    ],
    "balance_rmse_mol_L": 0.0037809930059328435,
    "minimum_predicted_concentration_mol_L": 0.00023598701015659085
  },
  "unscaled_output_model": {
    "rmse_mol_L": [
      0.006633520820218193,
      0.00391677353636534,
      0.004162037124132119
    ],
    "balance_rmse_mol_L": 0.00506280499988848,
    "minimum_predicted_concentration_mol_L": -0.002178128808736801
  },
  "decisio

Compare A/B/C errors in mol/L, concentration-sum residual, and minimum concentration for scaled and unscaled outputs. The stored seed-42 run has scaled-model RMSE about [0.00528,0.00294,0.00442] mol/L; environment changes may change the last digits. Inspect steady-training.png: validation, rather than the last epoch, chooses the saved weights. Change one setting using validation and then perform a final independent evaluation. The script also saves an optional operating-point preview for later weeks.

출력 scaling 유무에 따라 성분별 mol/L 오차, 농도 합 residual, 최소 농도를 비교합니다. 저장된 seed=42 실행의 scaled RMSE는 약 [0.00528,0.00294,0.00442] mol/L이며 환경에 따라 마지막 자릿수는 달라질 수 있습니다. steady-training.png에서 마지막 epoch 대신 validation으로 저장 가중치를 고르는 과정을 읽으세요. 설정 하나를 validation으로 비교한 뒤 독립 평가를 합니다. 스크립트는 후반 주차에서 다룰 운전점 선택 그림도 저장합니다.

## Report / 보고서

Explain a hand prediction, the corresponding computed intermediate values, and the final interpretation. Change one setting and compare your predicted effect with the observed result. Keep training, validation/calibration, and held-out evaluation roles separate.

손으로 예측한 값, 대응하는 중간 계산값, 최종 해석을 연결해 설명합니다. 설정 하나를 바꾸고 예상한 효과와 실제 결과를 비교합니다. Train·validation/calibration·독립 평가 데이터의 역할을 구분합니다.